# Worker de Colab para SoftSight

Convierte esta sesión en un **worker de renderizado de SoftSight**: el mismo puente local
(`tools/bridge.mjs`) se sirve por HTTP detrás de un túnel, y desde tu máquina le mandas
peticiones con el **mismo contrato JSON** que usarías por stdin. Solo cambia el transporte,
no la forma de la petición ni la de la respuesta.

**Antes de seguir, tres verdades:**
- Esta sesión es **efímera por diseño**: si se cae la pestaña o Google reasigna la VM,
  el trabajo en curso muere. Úsala para tandas, no como servidor permanente.
- SoftSight renderiza por **CPU** (sin GPU). La ganancia aquí son los **núcleos de CPU**
  y sacar el trabajo de tu máquina, no la tarjeta gráfica.
- El pool de `WORKER_CONCURRENCY` procesos `bridge.mjs` en paralelo es lo que escala:
  una petición por proceso.

**Orden:** config -> instalar Node -> obtener el código -> build -> worker -> túnel ->
usarlo desde el Mac. Si la sesión muere, reejecuta solo las celdas del worker y del túnel.


## 1. Configuración


In [ ]:
REPO_URL = "https://github.com/JesusGalindez/softsight.git"
TARGET = "/content/softsight"
WORKER_PORT = 8000
WORKER_CONCURRENCY = None            # None = núcleos del servidor (min 8); o un número
WORKER_CACHE_MB = 64                 # caché determinista en memoria del servidor
WORKER_TOKEN_OVERRIDE = ""         # vacío: se genera uno aleatorio
REPO_BRANCH = "fase-f-puesta-en-escena"  # rama que clona Colab; cambia si pusheas a main
MOUNTED = False                     # True si montaste el repo en TARGET


## 2. Instalar Node.js


In [ ]:
import os, sys, re, json, time, secrets, shutil, glob, subprocess, urllib.request

def run(args, cwd=None, check=False, timeout=1800):
    print(">>>", args if isinstance(args, str) else " ".join(args))
    p = subprocess.run(args, cwd=cwd, shell=isinstance(args, str), capture_output=True, text=True, timeout=timeout)
    if (p.stdout or "").strip(): print(p.stdout[-4000:], end="")
    if (p.stderr or "").strip() and p.returncode != 0: print(p.stderr[-2000:], file=sys.stderr, end="")
    if check and p.returncode != 0:
        raise SystemExit(f"fallo: {args} (código {p.returncode})")
    return p

def have(cmd):
    return shutil.which(cmd) is not None

if have("node") and have("npm"):
    print("node ya instalado:", subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip())
else:
    script = "export NVM_DIR=\"$HOME/.nvm\"; [ -s \"$NVM_DIR/nvm.sh\" ] && . \"$NVM_DIR/nvm.sh\"; nvm install 24 >/dev/null && nvm alias default 24 && node -v"
    ok = run(script)
    if ok.returncode != 0:
        print("sin nvm; bajo el binario oficial de Node...")
        run("curl -fsSL -o /tmp/node.tar.xz https://nodejs.org/dist/v24.13.0/node-v24.13.0-linux-x64.tar.xz", check=True)
        run("tar -xf /tmp/node.tar.xz -C /usr/local --strip-components=1", check=True)
    for name in ("node", "npm", "npx", "corepack"):
        if not have(name):
            candidates = glob.glob(os.path.expanduser("~/.nvm/versions/node/*/bin/") + name)
            if candidates:
                os.symlink(candidates[0], f"/usr/local/bin/{name}")
    if not have("node"):
        raise SystemExit("Node no quedó disponible")
print("node:", subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip())
print("npm:", subprocess.run(["npm", "--version"], capture_output=True, text=True).stdout.strip())


## 3. Obtener el código de SoftSight


In [ ]:
import os

if MOUNTED and os.path.exists(TARGET):
    print(f"usando la carpeta montada {TARGET}")
elif os.path.exists(os.path.join(TARGET, "tools", "bridge.mjs")):
    print(f"ya estaba el código en {TARGET}")
else:
    run(["git", "clone"] + (["-b", REPO_BRANCH] if REPO_BRANCH else []) + ["--depth", "1", REPO_URL, TARGET], check=True)
assert os.path.isfile(os.path.join(TARGET, "tools", "bridge.mjs")), "falta tools/bridge.mjs"


## 4. Compilar el CLI (dist-node)


In [ ]:
run(["npm", "run", "build:agent3d"], cwd=TARGET, check=True, timeout=1800)
assert os.path.isfile(os.path.join(TARGET, "dist-node", "agent3d.mjs")), "sin dist-node/agent3d.mjs"
print("build ok")


## 5. Arrancar el worker


In [ ]:
node_bin = shutil.which("node") or "/usr/bin/node"
token = WORKER_TOKEN_OVERRIDE or secrets.token_urlsafe(18)

log = open("/content/worker.log", "w")
env_w = dict(os.environ)
env_w.update({
    "WORKER_PORT": str(WORKER_PORT),
    "WORKER_HOST": "127.0.0.1",
    "WORKER_ACCESS_TOKEN": token,
    "WORKER_CACHE_MB": str(WORKER_CACHE_MB),
})
if WORKER_CONCURRENCY is not None:
    env_w["WORKER_CONCURRENCY"] = str(WORKER_CONCURRENCY)
proc = subprocess.Popen([node_bin, "tools/workerServer.mjs"], cwd=TARGET,
                       stdout=log, stderr=subprocess.STDOUT,
                       env=env_w, start_new_session=True)
open("/tmp/worker.pid", "w").write(str(proc.pid))

def health_ok():
    req = urllib.request.Request(f"http://127.0.0.1:{WORKER_PORT}/health",
                                headers={"Authorization": f"Bearer {token}"})
    try:
        with urllib.request.urlopen(req, timeout=3) as resp:
            return resp.status == 200
    except Exception:
        return False

print("esperando al worker...")
for _ in range(30):
    if health_ok():
        break
    time.sleep(1)
else:
    print(open("/content/worker.log").read()[-4000:])
    raise SystemExit("el worker no arrancó")
print("worker responde con token:", token[:6] + "...")
open("/content/worker.env", "w").write(f"SOFTSIGHT_WORKER_TOKEN={token}\n")


## 6. Exponer con un túnel de Cloudflare (quick tunnel)


In [ ]:
if not have("cloudflared"):
    run("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", check=True)
    os.chmod("/usr/local/bin/cloudflared", 0o755)

logfile = "/content/cloudflared.log"
open(logfile, "w").close()
subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{WORKER_PORT}",
                  "--no-autoupdate", "--logfile", logfile, "--loglevel", "info"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

url = None
for _ in range(50):
    time.sleep(3)
    text = open(logfile).read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if m:
        url = m.group(0)
        break
if not url:
    print(open(logfile).read()[-2000:])
    raise SystemExit("el túnel no dio URL")
with open("/content/worker.env", "a") as fh:
    fh.write(f"SOFTSIGHT_WORKER_URL={url}\n")
print("URL pública:", url)


## 7. Úsalo desde tu Mac


In [ ]:
env_text = open("/content/worker.env").read()
url = env_text.split("SOFTSIGHT_WORKER_URL=", 1)[1].split("\n", 1)[0]
token = env_text.split("SOFTSIGHT_WORKER_TOKEN=", 1)[1].split("\n", 1)[0]
print(f"""
Copia estas líneas y ejecútalas en tu Mac, en la raíz del repo softsight:

  export SOFTSIGHT_WORKER_URL="{url}"
  export SOFTSIGHT_WORKER_TOKEN="{token}"

Prueba rápida:
  node tools/workerClient.mjs --url $SOFTSIGHT_WORKER_URL --token $SOFTSIGHT_WORKER_TOKEN --health

UNA petición de render (los ficheros se piden por 'path'; el cliente los codifica local):
  node tools/workerClient.mjs --url $SOFTSIGHT_WORKER_URL --token $SOFTSIGHT_WORKER_TOKEN tools/worker-request-solo.json

Una tanda:
  node tools/workerClient.mjs --url $SOFTSIGHT_WORKER_URL --token $SOFTSIGHT_WORKER_TOKEN --batch peticion-a.json peticion-b.json

Notas:
- La URL es pública: cualquiera puede llegar. El token la protege (401 sin Bearer).
- Si la sesión muere, reejecuta las celdas 5 y 6 (worker y túnel) para una URL
  nueva; las celdas 2-4 no hacen falta.
""")


## 8. Notas finales

- El worker sirve el **mismo contrato** que `tools/bridge.mjs`: petición y respuesta
  son JSON idénticos. Un agente que escribía JSON por stdin solo cambia el transporte.
- Límites por defecto (se ajustan por env): petición hasta 32 MB, artefacto hasta 64 MB,
  timeout de trabajo 120 s, lote hasta 100.
- Lo que escala es la **cantidad de procesos `bridge.mjs` en paralelo**, no la GPU.
  Por defecto el pool usa los núcleos libres (máx 8); ajústalo con `WORKER_CONCURRENCY`.
- El servidor tiene **caché determinista**: la misma petición repetida se sirve desde
  memoria (SHA-256) sin tocar ningún proceso. Acotado por `WORKER_CACHE_MB`.
- No dejes el túnel abierto sin necesidad: al terminar la tanda, detén la sesión.
